# Tutorial 2a: metafspm writes and reads encoding3d

[metafspm](https://github.com/openalea/metafspm) is OpenAlea's framework for plant models made of **components**: one
for root growth, one for water flow, one for light... A **scene** grows several plants together. This tutorial shows
how a metafspm scene is saved as encoding3d files, and read back:

1. a small root growth component, written here for the tutorial (a toy, not a real model);
2. a scene of two plants, a frame recorded at each step;
3. those frames read with encoding3d alone, as any other platform would;
4. smaller frames, keeping only some scales or variables;
5. a frame turned back into metafspm's data structures, exactly as they were.

**What you need:** metafspm and encoding3d installed in the same environment (this folder's `README.md`).

In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

from openalea import encoding3d
from openalea.encoding3d.validate import validate_sequence
from openalea.metafspm import (actual, CompositeModel, parameter, Planting, planting, Scene, SceneRecorder,
                               segmentation, state_variable, StructuralComponent)
from openalea.metafspm.data_structure.configs import PropsConfig, ScalesConfig as scales
from openalea.metafspm.io.encoding3d import import_frame

OUTPUTS = "outputs"

## 1. A root growth component

A metafspm component declares its **variables** (with units, the scale they live at, their kind) and its
**processes**, the methods run at each time step. Ours grows one root axis: its tip elongates at a constant rate,
and when it is longer than 1 cm it is cut, a new tip continuing it. Each segment knows its two ends (`x1` … `z2`) and
its radius, so it can be drawn.

`initiate_plant` builds each plant's first segment, under the scales metafspm uses (plant, axis, growth unit,
phytomer, organ, segment), from the plant's seed (its `x`, `y`, `z`). The processes read the graph's values in bulk
(`values_at`) and write them back with `set_values`.

The plant model is a `CompositeModel`: its **parts** are its components (here one), run in order at each step.

In [ ]:
SEGMENT = 0.01                                           # m
UNIT = dict(unit_comment="", value_comment="", references="", DOI=[], min_value=None, max_value=None)


@dataclass
class ToyRootGrowth(StructuralComponent):
    """A root axis: the tip elongates at a constant rate, cut every 1 cm (a toy, for the tutorial)."""
    length: float = state_variable(unit="m", description="the segment's length", default=0., scale=scales.SubOrgan,
                                   kind="extensive", **UNIT)
    radius: float = state_variable(unit="m", description="the segment's radius", default=0.0005,
                                   scale=scales.SubOrgan, kind="intensive", **UNIT)
    is_apex: float = state_variable(unit="adim", description="1 for the tip", default=0., scale=scales.SubOrgan,
                                    kind="descriptor", **UNIT)
    x1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    y1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    z1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    x2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    y2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    z2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    elongation_rate: float = parameter(unit="m.s-1", description="the tip's elongation rate", default=2e-7,
                                       **UNIT)

    @classmethod
    def initiate_plant(cls, mtg, plant, scenario):
        s = mtg.scales
        axis = mtg.add_component(plant, **PropsConfig(scale=s.Axis, edge_type="/", label=mtg.labels.Axis.Root))
        unit = mtg.add_component(axis, **PropsConfig(scale=s.GrowthUnit, edge_type="/", label=mtg.labels.GrowthUnit.Root))
        phytomer = mtg.add_component(unit, **PropsConfig(scale=s.Phytomer, edge_type="/", label=mtg.labels.Phytomer.Root))
        organ = mtg.add_component(phytomer, **PropsConfig(scale=s.Organ, edge_type="/",
                                                        label=mtg.labels.Organ.RootInternode))
        tip = mtg.add_component(organ, **PropsConfig(scale=s.SubOrgan, edge_type="/", label=mtg.labels.SubOrgan.RootSegment))
        x, y, z = (float(mtg.values_at(name, [plant])[1][0]) for name in ("x", "y", "z"))    # the seed
        for name, value in (("length", 0.002), ("radius", 0.0005), ("is_apex", 1.), ("x1", x), ("y1", y), ("z1", z),
                            ("x2", x), ("y2", y), ("z2", z - 0.002)):
            mtg.set_values(name, [tip], [value])

    def _tips(self):
        """The tips' vertices, in order."""
        is_apex, present = self.mtg.column("is_apex")
        return np.flatnonzero(present & (is_apex == 1.))

    @actual
    def _elongation(self):
        """The tips grow downwards."""
        mtg, tips = self.mtg, self._tips()
        rate = dict(zip(self.data_structure.entity_ids("node").tolist(),
                        self.parameter_values("elongation_rate").tolist()))
        growth = np.array([rate[tip] for tip in tips.tolist()]) * self.step_duration
        mtg.set_values("length", tips, mtg.values_at("length", tips)[1] + growth)
        mtg.set_values("z2", tips, mtg.values_at("z2", tips)[1] - growth)

    @segmentation
    def _segmentation(self):
        """A tip longer than 1 cm becomes a segment, continued by a new tip."""
        mtg, tips = self.mtg, self._tips()
        tips = tips[mtg.values_at("length", tips)[1] > SEGMENT]
        if not tips.size:
            return
        rest = mtg.values_at("length", tips)[1] - SEGMENT
        radius, x, y = (mtg.values_at(name, tips)[1] for name in ("radius", "x2", "y2"))
        z = mtg.values_at("z1", tips)[1] - SEGMENT
        mtg.set_values("length", tips, SEGMENT)
        mtg.set_values("is_apex", tips, 0.)
        mtg.set_values("z2", tips, z)
        for k, tip in enumerate(tips.tolist()):
            mtg.add_child(tip, **PropsConfig(scale=mtg.scales.SubOrgan, edge_type="<", label=mtg.labels.SubOrgan.RootSegment,
                                           length=rest[k], is_apex=1., radius=radius[k],
                                           x1=x[k], y1=y[k], z1=z[k], x2=x[k], y2=y[k], z2=z[k] - rest[k]))


class ToyRoots(CompositeModel):
    """The plant model: one part, the root growth, run once per step."""
    initiators = (ToyRootGrowth,)
    parts = {"growth": ToyRootGrowth}

## 2. A scene, recorded

The **planting table** says which plants grow where: two plants 5 cm apart, their seeds 1 cm deep (`Planting`'s
`depth`, below the soil surface). `SceneRecorder(frame_period="step")` writes a frame at the start and after every
step, with the index `scene.fspm.json`: an encoding3d sequence, written in `frames/` while the scene runs. With
`pack_at_finish=True`, the run's end packs it into one file, `frames.fspm.zip`: the archive is checked (every file in
it, read back), then the folder removed. `simulate(8)` runs the eight steps, then finishes the run.

In [ ]:
table = planting([Planting(x=0.05 * i, depth=0.01) for i in range(2)], ToyRoots, x_range=0.2, y_range=0.1)
scene = Scene(planting=table, time_step="6 h",
              recorder=SceneRecorder(os.path.join(OUTPUTS, "full"), summaries=False, frame_period="step",
                                     pack_at_finish=True))
scene.simulate(8)                                      # two days, a step every six hours
archive = os.path.join(OUTPUTS, "full", "frames.fspm.zip")
report = validate_sequence(archive)
# warnings: metafspm also writes its graph's own properties (positions, types), which have no declared unit
print("valid:", report.ok, "|", len(report.errors), "errors,", len(report.warnings), "warnings")
print("the frames folder removed:", not os.path.exists(os.path.join(OUTPUTS, "full", "frames")))

## 3. The frames, read by encoding3d alone

Nothing in these files needs metafspm: any platform reads them, from the archive as from a folder. Each MPG scale is
an entity set, every variable a column with its unit, and the segments' triangles a mesh.

In [ ]:
rows = []
for frame in encoding3d.read_sequence(archive):
    plants = frame.data_structures[0]
    segments = plants.entity_set("SubOrgan")
    rows.append(dict(time_h=frame.time / 3600 if frame.time_unit == "s" else frame.time, step=frame.step,
                     segments=segments.count, total_length_cm=100 * segments.columns["length"].sum()))
print("entity sets:", [entity_set.name for entity_set in plants.entity_sets])
print("length's declaration:", segments.variables["length"])
pd.DataFrame(rows)

## 4. Smaller frames for logging

A long simulation writes many frames. `frame_options` keeps only what is needed: here the plants and the segments
(not the intermediate scales), and two variables, with the segments' ends that the frames' views draw them by. Such
frames are valid encoding3d files, for outputs only.

In [ ]:
scene = Scene(planting=table, time_step="6 h",
              recorder=SceneRecorder(os.path.join(OUTPUTS, "log"), summaries=False, frame_period="step",
                                     pack_at_finish=True,
                                     frame_options=dict(scales=["Plant", "SubOrgan"], variables=["length", "radius"],
                                                        geometry=False)))
scene.simulate(8)
log = os.path.join(OUTPUTS, "log", "frames.fspm.zip")
small = encoding3d.read(encoding3d.read_index(log)["frames"][-2]["path"])
print([entity_set.name for entity_set in small.data_structures[0].entity_sets],
      small.data_structures[0].entity_set("SubOrgan").variable_names())
print(f"full: {os.path.getsize(archive) / 1e3:.0f} kB, log: {os.path.getsize(log) / 1e3:.0f} kB")

## 5. Back into metafspm

A full frame rebuilds metafspm's data structures exactly: the same graph, the same variables, bit for bit. A
simulation can continue from it.

In [ ]:
last = encoding3d.read_index(archive)["frames"][-1]["path"]          # a frame inside the archive
rebuilt = import_frame(last)
population = scene.populations[0].name
data_structure = rebuilt[population]
print(population, "has", data_structure.n_nodes(), "segments; total length",
      round(100 * float(np.sum(data_structure.get("length"))), 3), "cm")

**In ParaView:** with the plugin loaded (tutorial 1), open `outputs/full/frames.fspm.zip` with the encoding3d reader
and press *Play*: the two roots grow.

## Try it yourself

- Change `elongation_rate` for one plant only, through its own scenario in the planting table
  (`planting(..., plant_scenarios=[{}, {"elongation_rate": 4e-7}])`).
- Record a frame every day (`frame_period="1 d"`) and compare the files.
- Keep the plants' scale only in the log, and read their number of segments from the full frames instead.